# Logistic Regression from Scratch: Gradient Descent on the Titanic Dataset

This notebook reimplements logistic regression example from Part 1, but instead of calling `LogisticRegression().fit()`,
we derive and implement **gradient descent** by hand.

We will:
1. Load and preprocess the Titanic dataset.
2. Review the mathematical foundations of logistic regression.
3. Implement gradient descent to learn the model weights.
4. Compare our results against scikit-learn.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


# sys.path.insert(0, '..')
# from src.logreg import logistic_regression_gd, predict, accuracy

## 1. Data Loading and Preprocessing

In [2]:
from mlops_track.data.paths import DATA_RAW

# data_folder = "../data/"
DATA_RAW_BIKE_RENTAL = DATA_RAW / "bike_rental"

booked_rentals_file_name = "registered_bike_rentals.csv"
direct_rentals_file_name = "direct_pickup_bike_rentals.csv"
weather_file_name = "weather.csv"
holiday_calendar_file_name = "holidays.csv"


def get_data_by_file_name(fileName: str, dir_name: str):
    full_path_name = dir_name / fileName
    df = pd.read_csv(full_path_name)
    return df

booked_rentals_data = get_data_by_file_name(booked_rentals_file_name, DATA_RAW_BIKE_RENTAL)
direct_rentals_data = get_data_by_file_name(direct_rentals_file_name, DATA_RAW_BIKE_RENTAL)
weather_data = get_data_by_file_name(weather_file_name, DATA_RAW_BIKE_RENTAL)
holiday_calendar_data = get_data_by_file_name(holiday_calendar_file_name, DATA_RAW_BIKE_RENTAL)

# print(booked_rentals_data.info())
# print(direct_rentals_data.info())
# print(weather_data.info())
# print(holiday_calendar_data.info())

print(booked_rentals_data.columns.tolist())
print(direct_rentals_data.columns.tolist())
print(weather_data.columns.tolist())
print(holiday_calendar_data.columns.tolist())

# print(booked_rentals_data.head())


# datas = (booked_rentals_data, direct_rentals_data, weather_data, holiday_calendar_data, )

# for file in datas:
#     df = pd.read_csv(file)
#     print(f"\nFile name: {file}")
#     print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    # print(f"\nMissing values:\n{df.isna().sum()}")
    # print(f"\nDuplicates: {df.duplicated().sum()}")
    # print("\n--- head ---")
    # print(df.head())    
    # print("\n--- info  ---")
    # print(df.info())
    # print("\n--- Descriptive Statistics ---")
    # print(df.describe())

['id', 'datetime', 'user_id', 'location_id']
['id', 'datetime', 'user_id', 'location_id']
['id', 'datetime', 'conditions', 'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh']
['id', 'date', 'holiday']


In [3]:
print(weather_data.head())
print(weather_data['conditions'].dropna().unique())


   id             datetime conditions  temperature_c  perceived_temperature_c  \
0   1  2011-01-01 00:00:00      clear            3.3                      3.0   
1   2  2011-01-01 01:00:00      clear            2.3                      2.0   
2   3  2011-01-01 02:00:00      clear            2.3                      2.0   
3   4  2011-01-01 03:00:00      clear            3.3                      3.0   
4   5  2011-01-01 04:00:00      clear            3.3                      3.0   

   humidity  windspeed_kmh  
0      81.0            0.0  
1      80.0            0.0  
2      80.0            0.0  
3      75.0            0.0  
4      75.0            0.0  
<StringArray>
['clear', 'clouds', 'light_rain', 'heavy_rain']
Length: 4, dtype: str


In [4]:
def process_rentals(booked_df, direct_df):
    """
    Step 1: Combine rental datasets, clean duplicates, and aggregate by hour.
    """
    # 1. Combine both rental datasets into one
    all_rentals = pd.concat([booked_df, direct_df], ignore_index=True)

    # 1.5 Data Cleaning: Check for and remove duplicate rows
    duplicates_count = all_rentals.duplicated().sum()
    print(f"   -> process_rentals: Found {duplicates_count} duplicate records. Dropping them...")
    all_rentals = all_rentals.drop_duplicates(ignore_index=True)

    # 2. Convert the datetime column to actual Pandas datetime objects
    col_name = 'datetime' 
    all_rentals[col_name] = pd.to_datetime(all_rentals[col_name])

    # 3. Floor the times to the nearest hour
    # Note for future: We use floor() instead of round() to create fixed "tumbling windows".
    # This matches industry standards for streaming data and prevents "date bleeding".
    all_rentals['hour'] = all_rentals[col_name].dt.floor('h')

    # 4. Count the number of rentals per hour
    hourly_rentals = all_rentals.groupby('hour').size().reset_index(name='total_rentals')
    
    print(f"   -> process_rentals: Aggregated data into {len(hourly_rentals)} business hours.")
    return hourly_rentals

In [5]:
def process_rentals_by_locId(booked_df, direct_df):
    """
    Step 1: Combine rental datasets, clean duplicates, and aggregate by hour.
    """
    # 1. Combine both rental datasets into one
    all_rentals = pd.concat([booked_df, direct_df], ignore_index=True)

    # 1.5 Data Cleaning: Check for and remove duplicate rows
    duplicates_count = all_rentals.duplicated().sum()
    print(f"   -> process_rentals: Found {duplicates_count} duplicate records. Dropping them...")
    all_rentals = all_rentals.drop_duplicates(ignore_index=True)

    # 2. Convert the datetime column to actual Pandas datetime objects
    col_name = 'datetime' 
    all_rentals[col_name] = pd.to_datetime(all_rentals[col_name])

    # 3. Floor the times to the nearest hour
    # Note for future: We use floor() instead of round() to create fixed "tumbling windows".
    # This matches industry standards for streaming data and prevents "date bleeding".
    all_rentals['hour'] = all_rentals[col_name].dt.floor('h')

    # 4. Count the number of rentals per hour
    hourly_rentals = all_rentals.groupby(['hour', 'location_id']).size().reset_index(name='total_rentals')
    # hourly_rentals = all_rentals.groupby(['hour', 'location_id'], sort=False).size().reset_index(name='total_rentals')
    # hourly_rentals = all_rentals.groupby(['location_id', 'hour']).size().reset_index(name='total_rentals')
    
    print(f"   -> process_rentals: Aggregated data into {len(hourly_rentals)} business hours.")
    return hourly_rentals

In [6]:
def merge_weather(hourly_rentals, weather_df):
    """
    Step 2: Merge the historical weather information with the hourly rentals.
    """
    # 1. We must make sure the weather datetime column is an actual datetime object
    weather_df = weather_df.copy()
    weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])
    
    # 2. Merge the datasets. 
    # We use a 'left' join to keep all hours even if weather is missing.
    merged_data = pd.merge(
        hourly_rentals, 
        weather_df, 
        left_on='hour', 
        right_on='datetime', 
        how='left'
    )
    
    # 3. Drop redundant columns. 
    # 'datetime' is redundant (we have 'hour'). 
    # 'id' is just an old database primary key from the weather CSV that has no predictive ML value.
    if 'datetime' in merged_data.columns:
        merged_data = merged_data.drop(columns=['datetime'])
    if 'id' in merged_data.columns:
        merged_data = merged_data.drop(columns=['id'])
    
    print(f"   -> merge_weather: Weather features added. Dataset now has {len(merged_data.columns)} columns.")
    
    return merged_data

In [7]:
def merge_holidays(data_with_weather, holidays_df):
    """
    Step 3: Merge the holiday calendar information.
    """
    holidays_df = holidays_df.copy()
    
    # 1. Convert the holiday 'date' column into a date object
    holidays_df['date'] = pd.to_datetime(holidays_df['date']).dt.date
    
    # 2. Extract just the date from our 'hour' column so we have a common key to match on
    data_with_weather['date'] = data_with_weather['hour'].dt.date
    
    # 3. Merge the datasets
    merged_data = pd.merge(
        data_with_weather, 
        holidays_df, 
        on='date', 
        how='left'
    )
    
    # 4. Fill missing holidays with a default value (like "no_holiday") since non-holidays don't appear in the CSV
    merged_data['holiday'] = merged_data['holiday'].fillna('no_holiday')
    
    # 5. Clean up the temporary 'date' column, and drop the useless 'id' from the holiday CSV
    columns_to_drop = ['date']
    if 'id' in merged_data.columns:
        columns_to_drop.append('id')
        
    merged_data = merged_data.drop(columns=columns_to_drop)
    
    print(f"   -> merge_holidays: Holiday features added. Dataset now has {len(merged_data.columns)} columns.")
    
    return merged_data

In [8]:
def derive_time_features(data_with_holidays):
    """
    Step 4: Derive additional time-based features (e.g., month, day of week)
    """
    final_dataset = data_with_holidays.copy()
    
    # Extract structural time components from our 'hour' datetime column
    final_dataset['month'] = final_dataset['hour'].dt.month
    final_dataset['day_of_week'] = final_dataset['hour'].dt.dayofweek
    # Note: day_of_week returns 0 for Monday, 6 for Sunday
    
    final_dataset['hour_of_day'] = final_dataset['hour'].dt.hour
    
    print(f"   -> derive_time_features: Time features derived. Final dataset has {len(final_dataset.columns)} columns.")
    
    return final_dataset

In [9]:
# Let's outline our main orchestration function that defines the pipeline's steps.
# We will define the helper functions for each step as we build them.

def run_preprocessing_pipeline(booked_df, direct_df, weather_df, holidays_df):
    print("Starting data preprocessing pipeline...")
    
    # Step 1: Load, clean, and combine the operational data (aggregate to hourly)
    # final_dataset = process_rentals(booked_df, direct_df)
    # final_dataset = process_rentals_by_locId(booked_df, direct_df)
    
    # Step 2: Merge the historical weather information
    # final_dataset = merge_weather(final_dataset, weather_df)
    
    # Step 3: Merge the holiday calendar information
    # final_dataset = merge_holidays(final_dataset, holidays_df)
    
    # Step 4: Derive additional time-based features (e.g., month, day of week)
    # final_dataset = derive_time_features(final_dataset)
    
    final_dataset = (
        # process_rentals(booked_df, direct_df)
        process_rentals_by_locId(booked_df, direct_df)
        .pipe(merge_weather, weather_df=weather_df)
        .pipe(merge_holidays, holidays_df=holidays_df)
        .pipe(derive_time_features)
    )
    print("Pipeline finished successfully!")
    return final_dataset

final_data = run_preprocessing_pipeline(booked_rentals_data, direct_rentals_data, weather_data, holiday_calendar_data)
final_data.head()

Starting data preprocessing pipeline...
   -> process_rentals: Found 0 duplicate records. Dropping them...
   -> process_rentals: Aggregated data into 312386 business hours.
   -> merge_weather: Weather features added. Dataset now has 8 columns.
   -> merge_holidays: Holiday features added. Dataset now has 9 columns.
   -> derive_time_features: Time features derived. Final dataset has 12 columns.
Pipeline finished successfully!


,hour,location_id,total_rentals,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,holiday,month,day_of_week,hour_of_day
0,2011-01-01,2,1,clear,3.3,3.0,81.0,0.0,no_holiday,1,5,0
1,2011-01-01,3,1,clear,3.3,3.0,81.0,0.0,no_holiday,1,5,0
2,2011-01-01,4,2,clear,3.3,3.0,81.0,0.0,no_holiday,1,5,0
3,2011-01-01,5,1,clear,3.3,3.0,81.0,0.0,no_holiday,1,5,0
4,2011-01-01,9,1,clear,3.3,3.0,81.0,0.0,no_holiday,1,5,0


In [10]:
def my_test(booked_df, direct_df):
    all_rentals = pd.concat([booked_df, direct_df], ignore_index=True)
    all_rentals = all_rentals.drop_duplicates(ignore_index=True)
    all_rentals['datetime'] = pd.to_datetime(all_rentals['datetime'])
    all_rentals['hour'] = all_rentals['datetime'].dt.floor('h')
    # Try the user's code exact
    hourly_rentals = all_rentals.groupby(['location_id', 'hour']).size().reset_index(name='total_rentals')
    return hourly_rentals

# my_test(booked_rentals_data, direct_rentals_data)


In [11]:
# Filter the dataset for location 0 and date 2011-01-01
filtered_data = final_data[
    (final_data['location_id'] == 0) &
    (final_data['hour'].dt.date == pd.to_datetime('2011-01-01').date())
]
print(filtered_data)


                   hour  location_id  total_rentals  conditions  \
11  2011-01-01 01:00:00            0              1       clear   
28  2011-01-01 02:00:00            0              1       clear   
43  2011-01-01 03:00:00            0              2       clear   
59  2011-01-01 08:00:00            0              1       clear   
66  2011-01-01 09:00:00            0              3       clear   
75  2011-01-01 10:00:00            0              4       clear   
91  2011-01-01 11:00:00            0              1       clear   
111 2011-01-01 12:00:00            0              2       clear   
131 2011-01-01 13:00:00            0              1      clouds   
152 2011-01-01 14:00:00            0             10      clouds   
173 2011-01-01 15:00:00            0              6      clouds   
194 2011-01-01 16:00:00            0              6      clouds   
233 2011-01-01 18:00:00            0              2  light_rain   
253 2011-01-01 19:00:00            0              3  light_rai

In [12]:
def basic_audit_dataframe_detailed(df: pd.DataFrame, name: str) -> None:
    """Basic structural information for a DataFrame.

    Reports shape, full-row duplicates, and a per-column summary table
    with type, range/nunique, and null counts. For low-cardinality
    categorical columns (<= 25 unique values) full value_counts are
    listed below. First 5 rows are appended at the end for a sanity check.
    """
    print(f"Dataset:  {name}\n")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Duplicate rows: {df.duplicated().sum()}")

    def thousands(n) -> str:
        return f"{int(n):,}".replace(",", " ")

    total_rows = len(df)
    rows = []
    low_card = []  # (col, value_counts) for non-numeric, non-date columns with <= 25 unique values

    for col in df.columns:
        s = df[col]
        nulls = int(s.isna().sum())
        is_int   = pd.api.types.is_integer_dtype(s)
        is_float = pd.api.types.is_float_dtype(s)
        name_is_date = ("date" in col.lower()) or ("time" in col.lower())

        if is_int:
            type_label = "int"
            nu = s.nunique()
            mn, mx = thousands(s.min()), thousands(s.max())
            range_str = f"{mn}..{mx}, unique" if nu == total_rows else f"{mn}..{mx} ({thousands(nu)} unique)"
        elif is_float:
            type_label = "float"
            nu = s.nunique()
            range_str = f"{s.min()}..{s.max()} ({thousands(nu)} unique)"
        elif name_is_date:
            type_label = "str"
            ts = pd.to_datetime(s, errors="coerce")
            if ts.notna().any():
                mn = ts.min().strftime("%Y-%m-%d %H:%M")
                mx = ts.max().strftime("%Y-%m-%d %H:%M")
                range_str = f"{mn} → {mx}"
            else:
                range_str = f"{thousands(s.nunique())} unique (parse failed)"
        else:
            type_label = "str"
            nu = s.nunique()
            if nu <= 25:
                low_card.append((col, s.value_counts(dropna=False)))
                range_str = f"{thousands(nu)} unique (see value_counts below)"
            else:
                range_str = f"{thousands(nu)} unique"

        rows.append((col, type_label, range_str, nulls))

    col_w  = max(len("Column"),          max(len(r[0]) for r in rows))
    typ_w  = max(len("Type"),            max(len(r[1]) for r in rows))
    rng_w  = max(len("Range / nunique"), max(len(r[2]) for r in rows))
    null_w = max(len("Missing values"),  max(len(str(r[3])) for r in rows))

    print("\n--- Per-column summary ---")
    print(
        f"| {'Column':<{col_w}} | {'Type':<{typ_w}} | "
        f"{'Range / nunique':<{rng_w}} | {'Missing values':>{null_w}} |"
    )
    print(
        f"|{'-' * (col_w + 2)}|{'-' * (typ_w + 2)}|"
        f"{'-' * (rng_w + 2)}|{'-' * (null_w + 2)}|"
    )
    for c, t, r, n in rows:
        print(
            f"| {c:<{col_w}} | {t:<{typ_w}} | "
            f"{r:<{rng_w}} | {n:>{null_w}} |"
        )

    for col, vc in low_card:
        print(f"\nvalue_counts of [{col}]:")
        for val, cnt in vc.items():
            print(f"  {val!r}: {thousands(cnt)}")

    print("\nFirst 5 rows:")
    print(df.head().to_string())


### 6.1 Hourly rental aggregation

Both rental sources record one row per individual rental event. Here we aggregate them to **counts per hour per location**, producing `registered_rentals`, `direct_pickups`, and `total_rentals`.

We also complete the grid: every `(hour, location)` pair in the operational window gets a row, with zeros where no rentals happened. Without this, "quiet" pairs would be silently missing from the final dataset and the model would never learn the "no demand" pattern at low-activity kiosks. The result matches the hourly grain of the weather data.


In [13]:
rentals = pd.concat([
    registered_rentals_data.assign(is_registered=True),
    direct_rentals_data.assign(is_registered=False),
], ignore_index=True)
rentals["datetime_hourly"] = pd.to_datetime(rentals["datetime"]).dt.floor("h")

hourly_counts = (
    rentals.groupby(["datetime_hourly", "location_id", "is_registered"])
    .size().unstack(fill_value=0)
    .rename(columns={True: "registered_rentals", False: "direct_pickups"})
    .rename_axis(columns=None).reset_index()
    [["datetime_hourly", "location_id", "registered_rentals", "direct_pickups"]]
)

full_grid = pd.MultiIndex.from_product(
    [sorted(hourly_counts["datetime_hourly"].unique()),
     sorted(hourly_counts["location_id"].unique())],
    names=["datetime_hourly", "location_id"],
).to_frame(index=False)

hourly_rentals = full_grid.merge(
    hourly_counts, on=["datetime_hourly", "location_id"], how="left"
)
for col in ["registered_rentals", "direct_pickups"]:
    hourly_rentals[col] = hourly_rentals[col].fillna(0).astype(int)

hourly_rentals["total_rentals"] = hourly_rentals["registered_rentals"] + hourly_rentals["direct_pickups"]

print(f"Shape: {hourly_rentals.shape}")
print(hourly_rentals.head().to_string(index=False))

NameError: name 'registered_rentals_data' is not defined